# Gold Layer Aggregations — ZOSA Lakehouse
Business-ready tables for reporting and analytics.

Each Gold table combines, aggregates, or enriches Silver data
to answer specific business questions.

In [ ]:
from pyspark.sql.functions import (
    col, year, month, count, sum as spark_sum, avg, max as spark_max,
    when, round as spark_round, lit, date_format
)

## gold_asteroid_risk
Compute a composite hazard score and assign a risk category to each asteroid.

In [ ]:
asteroids = spark.table("asteroids_silver")

# Hazard score: weighted combination of size, speed, and proximity
gold_risk = asteroids.withColumn(
    "hazard_score",
    spark_round(
        (col("diameter_km") * 0.4) +
        (col("velocity_km_s") * 0.35) +
        ((1.0 / (col("miss_distance_au") + 0.001)) * 0.25),
        4
    )
).withColumn(
    "risk_category",
    when(col("hazard_score") >= 50, "Critical")
    .when(col("hazard_score") >= 25, "High")
    .when(col("hazard_score") >= 10, "Medium")
    .otherwise("Low")
)

gold_risk.write.mode("overwrite").format("delta").saveAsTable("gold_asteroid_risk")
print(f"✅ gold_asteroid_risk: {gold_risk.count()} rows")
gold_risk.groupBy("risk_category").count().show()

## gold_mission_summary
Aggregate missions by year, type, status, and region for executive dashboards.

In [ ]:
missions = spark.table("missions_silver")

gold_missions = missions.withColumn("launch_year", year(col("launch_date"))) \
    .groupBy("launch_year", "mission_type", "status", "region") \
    .agg(
        count("*").alias("mission_count"),
        spark_round(spark_sum("budget_millions"), 2).alias("total_budget_millions"),
        spark_round(avg("budget_millions"), 2).alias("avg_budget_millions")
    ) \
    .orderBy("launch_year", "mission_type")

gold_missions.write.mode("overwrite").format("delta").saveAsTable("gold_mission_summary")
print(f"✅ gold_mission_summary: {gold_missions.count()} rows")

## gold_solar_activity
Monthly timeline of solar events with counts by type and peak intensity.

In [ ]:
solar = spark.table("solar_events_silver")

gold_solar = solar.withColumn("event_month", date_format(col("event_date"), "yyyy-MM")) \
    .groupBy("event_month", "event_type") \
    .agg(
        count("*").alias("event_count"),
        spark_round(spark_max("intensity"), 2).alias("peak_intensity"),
        spark_round(avg("duration_hours"), 2).alias("avg_duration_hours")
    ) \
    .orderBy("event_month")

gold_solar.write.mode("overwrite").format("delta").saveAsTable("gold_solar_activity")
print(f"✅ gold_solar_activity: {gold_solar.count()} rows")

## gold_exoplanet_catalog
Filter for habitable-zone candidates and compute a simplified Earth Similarity Index (ESI).

In [ ]:
exo = spark.table("exoplanets_silver")

# Habitable zone: equilibrium temp roughly 200-320 K
gold_exo = exo.withColumn(
    "in_habitable_zone",
    when((col("equilibrium_temp_k") >= 200) & (col("equilibrium_temp_k") <= 320), True)
    .otherwise(False)
).withColumn(
    # Simplified ESI based on radius and temperature similarity to Earth
    "earth_similarity_index",
    spark_round(
        1.0 - (
            (abs(col("radius_earth") - 1.0) / 3.0) * 0.5 +
            (abs(col("equilibrium_temp_k") - 288.0) / 400.0) * 0.5
        ),
        4
    )
)

gold_exo.write.mode("overwrite").format("delta").saveAsTable("gold_exoplanet_catalog")
print(f"✅ gold_exoplanet_catalog: {gold_exo.count()} rows")
print(f"   Habitable zone candidates: {gold_exo.filter(col('in_habitable_zone')).count()}")

## Validation

In [ ]:
gold_tables = ["gold_asteroid_risk", "gold_mission_summary",
               "gold_solar_activity", "gold_exoplanet_catalog"]

for tbl in gold_tables:
    cnt = spark.sql(f"SELECT COUNT(*) AS cnt FROM {tbl}").collect()[0]["cnt"]
    print(f"  {tbl}: {cnt} rows")

print("\n✅ Gold layer complete — ready for Power BI and analytics.")